In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

analysis_dir = "/accounts/projects/jchayes/commnotes/commnotes-analysis/analysis"
os.chdir(analysis_dir)
sys.path.insert(0, analysis_dir)

In [3]:
import importlib
import src.helper as helper

importlib.reload(helper)

<module 'src.helper' from '/accounts/projects/jchayes/commnotes/commnotes-analysis/analysis/src/helper.py'>

In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from linearmodels.panel import PanelOLS
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde, spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess

from src.data_loader import *
from src.analysis import *
from src.utils import *
from src.visualization import *
from src.visualization import (
    apply_publication_style,
    _setup_ax,
    _finalize,
    _C_BLUE,
    _C_ORANGE,
    _C_EVENT,
    _C_ZERO,
    _C_LIGHT,
    _C_PURPLE,
    _C_AMBER,
    _MUTED_PALETTE,
    _FONT_LEGEND,
    _FONT_ANNOT,
    _FONT_TICK,
    _FONT_LABEL,
)

In [5]:
# --- Configuration (Corrected Paths) ---
# Using the absolute paths from your working environment
NOTES_PATH = "/scratch/users/commnotes/communitynotes/sourcecode/notes-00000.tsv"
HISTORY_PATH = "/scratch/users/commnotes/communitynotes/sourcecode/noteStatusHistory-00000.tsv"
RATINGS_DIR = "/scratch/users/commnotes/communitynotes/sourcecode/ratings"
WEEKLY_OUTPUTS_PATH = "/scratch/users/commnotes/communitynotes2022/communitynotes/static/sourcecode/weekly_outputs"
WEEKLY_OUTPUTS_PATH_2025 = "/scratch/users/commnotes/communitynotes/sourcecode/weekly_outputs"


start_date = "2022-06-01"
max_date = "2023-06-01"

In [6]:
print("--- Loading and Preparing Data ---")
# This DataFrame contains data related to "notes" associated with tweets. It includes unique IDs for notes, authors, and tweets, along with
# timestamps, classifications (e.g., "MISINFORMED_OR_POTENTIALLY_MISLEADING"), and other
# attributes describing the history of the notes
notes_df = pd.read_csv(NOTES_PATH, sep='\\t', dtype={'noteAuthorParticipantId': str, 'tweetId': str, 'noteId': str})
notes_df["noteId"]=notes_df["noteId"].astype(int)

# The following is all note ratings published by X keeping the following columns: 
# "'noteId', 'raterParticipantId', 'createdAtMillis', 'helpfulnessLevel'" since we don't use the rest.
ratings_df = read_ratings_from_directory(RATINGS_DIR, start_date_str="2020-06-01",end_date_str=max_date)

# The following df is the results of the matrix factorization for both 2022 and 2025 version of the code.
note_df_full, rater_df, status_df, helpfulness_df, aux_df = load_weekly_outputs(WEEKLY_OUTPUTS_PATH,debug=False)
note_df_full_2025, rater_df_2025, status_df_2025, helpfulness_df_2025, aux_df_2025 = load_weekly_outputs(WEEKLY_OUTPUTS_PATH_2025,debug=False)

# This df is the history of all notes
history_df =   pd.read_csv(HISTORY_PATH, sep='\\t', dtype={'noteAuthorParticipantId': str, 'tweetId': str, 'noteId': str})

# The following df is notes with corresponding topics. Topics are modelled by the X topic modeling algorithm.
notes_df_topic = pd.read_csv("/accounts/projects/jchayes/commnotes/df_notes_with_topics.csv")

# The following df is notes with corresponding topics based on LLM topic modelers.
notes_df_llms = pd.read_csv("/accounts/projects/jchayes/commnotes/notes_llm_topic.csv")


notes_df_topic['noteId'] = notes_df_topic['noteId'].astype(int)

--- Loading and Preparing Data ---


/tmp/ipykernel_1643931/1959131124.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.


Found 16 total rating files. Processing them in chunks to filter by date.


Loading weekly outputs: 100%|██████████| 84/84 [02:01<00:00,  1.45s/it]


note is loaded.
rater is loaded.
status is loaded.
helpfulness is loaded.
aux is loaded.


Loading weekly outputs: 100%|██████████| 44/44 [00:02<00:00, 15.79it/s]
/tmp/ipykernel_1643931/1959131124.py:17: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.


note is loaded.
rater is loaded.
status is loaded.
key without data status
helpfulness is loaded.
key without data helpfulness
aux is loaded.
key without data aux


In [7]:
note_df_full, rater_df, status_df, helpfulness_df, aux_df = load_weekly_outputs(WEEKLY_OUTPUTS_PATH,debug=False)
note_df_full_2025, rater_df_2025, status_df_2025, helpfulness_df_2025, aux_df_2025 = load_weekly_outputs(WEEKLY_OUTPUTS_PATH_2025,debug=False)

Loading weekly outputs:   0%|          | 0/84 [00:00<?, ?it/s]

Loading weekly outputs: 100%|██████████| 84/84 [02:07<00:00,  1.52s/it]


note is loaded.
rater is loaded.
status is loaded.
helpfulness is loaded.
aux is loaded.


Loading weekly outputs: 100%|██████████| 44/44 [00:02<00:00, 16.38it/s]

note is loaded.
rater is loaded.
status is loaded.
key without data status
helpfulness is loaded.
key without data helpfulness
aux is loaded.
key without data aux


In [8]:
xbow_df = notes_df_topic[['noteId', 'noteTopic']]
notes_df_llms['LLAMA4'] = helper.clean_topic_series(notes_df_llms['LLAMA4'])
notes_df_llms['TOPIC'] = helper.clean_topic_series(notes_df_llms['TOPIC'])

In [62]:
cols = ['noteId', 'createdAtMillis', 'tweetId', 'summary']
notes_df_filter = notes_df[cols]

In [63]:
CONTROVERSIAL_TOPICS = [
    'USPolitics', 'UkraineConflict', 'GazaConflict',
    'CrimeLegal', 'HealthCovid', 'EconomyFinance','MessiRonaldo',  'politics', 'economy', 'health'
]

NON_CONTROVERSIAL_TOPICS = [
    'SpaceAstronomy', 'ClimateEnvironment', 'EntertainmentMoviesTV',
    'WeatherDisasters', 'TechCompanies', 'SportsNFL', 'SportsNBA',
'FoodNutrition', 'Education', 'Scams',
    'ArtificialIntelligence', 'science', 'other'
]

In [64]:
xbow_topics = (
    xbow_df[["noteId", "noteTopic"]]
    .rename(columns={"noteTopic": "xbow_topic"})
)

llm_topics = (
    notes_df_llms[["NOTEID", "TOPIC", "LLAMA4"]]
    .rename(columns={
        "NOTEID": "noteId",
        "TOPIC": "topic",
        "LLAMA4": "llama",
    })
)

notes_df_filter = (
    notes_df_filter
    .merge(xbow_topics, on="noteId", how="left")
    .merge(llm_topics, on="noteId", how="left")
)

In [65]:
notes_df_filter.head()

,noteId,createdAtMillis,tweetId,summary,xbow_topic,topic,llama
0,1783179305159200982,1713978050878,1783159712986382830,The House failed to pass a border protection l...,USPolitics,USPolitics,USPolitics
1,1783181538789605871,1713978583415,1783171851818021181,The United States has 50 States https://da...,NaN,USPolitics,USPolitics
2,1783182562279494134,1713978827435,1783154445682979015,TikTok only mentions “ban” and chooses to igno...,NaN,TechCompanies,TechCompanies
3,1883711635770196070,1737946826294,1883619411774345444,This could be considered a threat https://...,Scams,<NA>,<NA>
4,1537142913737428992,1655318404027,1377030478167937024,Forbes has a good rundown of the investigation...,USPolitics,CrimeLegal,USPolitics


# Add Factor-Based Controversy

In [66]:
factor_controversy_df = helper.compute_factor_based_controversy(note_df_full)

notes_df_filter = notes_df_filter.merge(
    factor_controversy_df,
    on="noteId",
    how="left",
)

In [67]:
CONTROVERSIAL_TOPICS_SET = set(CONTROVERSIAL_TOPICS)
NON_CONTROVERSIAL_TOPICS_SET = set(NON_CONTROVERSIAL_TOPICS)


def map_topic_to_controversy(topic):
    if pd.isna(topic):
        return np.nan
    if topic in CONTROVERSIAL_TOPICS_SET:
        return 1
    if topic in NON_CONTROVERSIAL_TOPICS_SET:
        return 0
    return np.nan

In [68]:
topic_cols = ["xbow_topic", "topic", "llama"]

for col in topic_cols:
    notes_df_filter[f"{col}_controversy"] = notes_df_filter[col].apply(
        map_topic_to_controversy
    )

In [69]:
notes_df_filter.head()

,noteId,createdAtMillis,tweetId,summary,xbow_topic,topic,llama,factor_controversy,xbow_topic_controversy,topic_controversy,llama_controversy
0,1783179305159200982,1713978050878,1783159712986382830,The House failed to pass a border protection l...,USPolitics,USPolitics,USPolitics,NaN,1.0,1.0,1.0
1,1783181538789605871,1713978583415,1783171851818021181,The United States has 50 States https://da...,NaN,USPolitics,USPolitics,NaN,NaN,1.0,1.0
2,1783182562279494134,1713978827435,1783154445682979015,TikTok only mentions “ban” and chooses to igno...,NaN,TechCompanies,TechCompanies,NaN,NaN,0.0,0.0
3,1883711635770196070,1737946826294,1883619411774345444,This could be considered a threat https://...,Scams,<NA>,<NA>,NaN,0.0,NaN,NaN
4,1537142913737428992,1655318404027,1377030478167937024,Forbes has a good rundown of the investigation...,USPolitics,CrimeLegal,USPolitics,NaN,1.0,1.0,1.0


In [70]:
# write to CSV
CONTROVERSY_PATH = "/accounts/projects/jchayes/commnotes/commnotes-analysis/analysis/notes_by_controversy.csv"
notes_df_filter.to_csv(CONTROVERSY_PATH, index=False)

In [71]:
notes_by_controversy = pd.read_csv(CONTROVERSY_PATH)

In [72]:
notes_by_controversy.head()

,noteId,createdAtMillis,tweetId,summary,xbow_topic,topic,llama,factor_controversy,xbow_topic_controversy,topic_controversy,llama_controversy
0,1783179305159200982,1713978050878,1783159712986382830,The House failed to pass a border protection l...,USPolitics,USPolitics,USPolitics,NaN,1.0,1.0,1.0
1,1783181538789605871,1713978583415,1783171851818021181,The United States has 50 States https://da...,NaN,USPolitics,USPolitics,NaN,NaN,1.0,1.0
2,1783182562279494134,1713978827435,1783154445682979015,TikTok only mentions “ban” and chooses to igno...,NaN,TechCompanies,TechCompanies,NaN,NaN,0.0,0.0
3,1883711635770196070,1737946826294,1883619411774345444,This could be considered a threat https://...,Scams,NaN,NaN,NaN,0.0,NaN,NaN
4,1537142913737428992,1655318404027,1377030478167937024,Forbes has a good rundown of the investigation...,USPolitics,CrimeLegal,USPolitics,NaN,1.0,1.0,1.0


# Summary

In [73]:
classifier_cols = ["xbow_topic", "topic", "llama"]

# A) Missing entries for each classifier
missing_counts = notes_df_filter[classifier_cols].isna().sum()
missing_rates = notes_df_filter[classifier_cols].isna().mean()

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_rate": missing_rates,
})

display(missing_summary)


# B) Agreement only among rows where all classifiers have a label
labeled = notes_df_filter.dropna(subset=classifier_cols).copy()

all_agree_count = (
    (labeled["xbow_topic"] == labeled["topic"]) &
    (labeled["topic"] == labeled["llama"])
).sum()

all_labeled_count = len(labeled)
all_agree_rate = all_agree_count / all_labeled_count if all_labeled_count else float("nan")

agreement_summary = pd.DataFrame({
    "all_labeled_count": [all_labeled_count],
    "all_agree_count": [all_agree_count],
    "all_agree_rate": [all_agree_rate],
})

display(agreement_summary)

,missing_count,missing_rate
xbow_topic,94013,0.642161
topic,42144,0.287867
llama,45546,0.311104


,all_labeled_count,all_agree_count,all_agree_rate
0,37916,19906,0.525003


In [74]:
pairs = [
    ("xbow_topic", "topic"),
    ("xbow_topic", "llama"),
    ("topic", "llama"),
]

rows = []

# counts agreement divided by rows where a label exists for each classifier in the pair
for a, b in pairs:
    pair_labeled = notes_df_filter[[a, b]].notna().all(axis=1)
    agree = notes_df_filter.loc[pair_labeled, a] == notes_df_filter.loc[pair_labeled, b]

    rows.append({
        "comparison": f"{a} vs {b}",
        "pair_labeled_count": pair_labeled.sum(),
        "agree_count": agree.sum(),
        "agree_rate": agree.mean(),
    })

pairwise_agreement = pd.DataFrame(rows)

display(pairwise_agreement)

,comparison,pair_labeled_count,agree_count,agree_rate
0,xbow_topic vs topic,38686,21849,0.564778
1,xbow_topic vs llama,38056,21331,0.560516
2,topic vs llama,99894,74495,0.745740


In [75]:
classifier_cols = ["xbow_topic", "topic", "llama"]

# Count agreement only when all three are present and equal.
all_three_present = notes_df_filter[classifier_cols].notna().all(axis=1)

all_three_agree_no_missing = (
    all_three_present &
    notes_df_filter[classifier_cols].nunique(axis=1, dropna=True).eq(1)
)

overall_agree_count = all_three_agree_no_missing.sum()
total_notes = len(notes_df_filter)
overall_agree_rate = overall_agree_count / total_notes if total_notes else float("nan")

overall_agreement_summary = pd.DataFrame({
    "total_notes": [total_notes],
    "overall_agree_count": [overall_agree_count],
    "overall_agree_rate": [overall_agree_rate],
    "overall_agree_percent": [overall_agree_rate * 100],
})

display(overall_agreement_summary)

,total_notes,overall_agree_count,overall_agree_rate,overall_agree_percent
0,146401,19906,0.135969,13.596902


# Create Dataframe

In [84]:
notes_df_filter.head()

,noteId,createdAtMillis,tweetId,summary,xbow_topic,topic,llama,factor_controversy,xbow_topic_controversy,topic_controversy,llama_controversy,createdAtDate
0,1783179305159200982,1713978050878,1783159712986382830,The House failed to pass a border protection l...,USPolitics,USPolitics,USPolitics,NaN,1.0,1.0,1.0,2024-04-24 17:00:50.878
1,1783181538789605871,1713978583415,1783171851818021181,The United States has 50 States https://da...,NaN,USPolitics,USPolitics,NaN,NaN,1.0,1.0,2024-04-24 17:09:43.415
2,1783182562279494134,1713978827435,1783154445682979015,TikTok only mentions “ban” and chooses to igno...,NaN,TechCompanies,TechCompanies,NaN,NaN,0.0,0.0,2024-04-24 17:13:47.435
3,1883711635770196070,1737946826294,1883619411774345444,This could be considered a threat https://...,Scams,<NA>,<NA>,NaN,0.0,NaN,NaN,2025-01-27 03:00:26.294
4,1537142913737428992,1655318404027,1377030478167937024,Forbes has a good rundown of the investigation...,USPolitics,CrimeLegal,USPolitics,NaN,1.0,1.0,1.0,2022-06-15 18:40:04.027


In [88]:
controversy_classifier_cols = [
    "factor_controversy",
    "xbow_topic_controversy",
    "topic_controversy",
    "llama_controversy",
]

l_dfs = {}
p_dfs = {}

for classifier_col in controversy_classifier_cols:
    notes_for_helper = (
        notes_df_filter[
            ["noteId", "createdAtMillis", "tweetId", classifier_col]
        ]
        .rename(columns={classifier_col: "controversial"})
        .dropna(subset=["controversial"])
        .copy()
    )

    print("\n", classifier_col)
    print(notes_for_helper["controversial"].value_counts(dropna=False))

    l_df, p_df = helper.calculate_controversy_proportions_principled(
        status_history_df=history_df,
        notes_df=notes_for_helper,
        time="W",
        min_n=30,
        smooth_window=7,
    )

    l_dfs[classifier_col] = l_df
    p_dfs[classifier_col] = p_df.assign(classifier=classifier_col)


 factor_controversy
controversial
0.0    1270
1.0     318
Name: count, dtype: int64

 xbow_topic_controversy
controversial
1.0    36818
0.0    15570
Name: count, dtype: int64

 topic_controversy
controversial
1.0    43560
0.0    24965
Name: count, dtype: int64

 llama_controversy
controversial
1.0    45409
0.0    23025
Name: count, dtype: int64


# Pre vs. Post Number of Tweets With One (C/NC) Note Rated Helpful (plus CI)

In [106]:
def wilson_ci(k, n, z=1.96):
    k = np.asarray(k, dtype=float)
    n = np.asarray(n, dtype=float)

    with np.errstate(divide="ignore", invalid="ignore"):
        p = k / n
        denom = 1 + z**2 / n
        center = (p + z**2 / (2 * n)) / denom
        margin = (
            z
            * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
            / denom
        )

    lo = center - margin
    hi = center + margin

    return lo, hi

In [117]:
def tweet_topic_share_pre_post(
    notes_df,
    controversy_col,
    cutoff_date="2022-10-01",
    window_weeks=12,
    date_col="createdAtDate",
):
    cutoff = pd.to_datetime(cutoff_date)
    pre_start = cutoff - pd.Timedelta(weeks=window_weeks)
    post_end = cutoff + pd.Timedelta(weeks=window_weeks)

    df = notes_df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    df = df[
        (df[date_col] >= pre_start)
        & (df[date_col] < post_end)
    ].copy()

    df["period"] = np.where(df[date_col] < cutoff, "pre", "post")

    total_tweets = (
        df
        .groupby("period")["tweetId"]
        .nunique()
        .rename("total_tweets")
    )

    classified = df.dropna(subset=[controversy_col]).copy()

    topic_tweets = (
        classified
        .groupby(["period", controversy_col])["tweetId"]
        .nunique()
        .rename("n_tweets")
        .reset_index()
    )

    out = topic_tweets.merge(total_tweets, on="period", how="left")

    out["topic_type"] = out[controversy_col].map({
        1: "controversial",
        0: "non_controversial",
    })

    out["proportion"] = out["n_tweets"] / out["total_tweets"]
    out["percent"] = 100 * out["proportion"]

    out["ci_lo"], out["ci_hi"] = wilson_ci(
        out["n_tweets"],
        out["total_tweets"],
    )

    out["ci_lo_percent"] = 100 * out["ci_lo"]
    out["ci_hi_percent"] = 100 * out["ci_hi"]

    wide = (
        out
        .pivot(index="topic_type", columns="period", values=[
            "percent",
            "ci_lo_percent",
            "ci_hi_percent",
        ])
        .reset_index()
    )

    wide.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col
        for col in wide.columns
    ]

    wide["post_minus_pre_pp"] = wide["percent_post"] - wide["percent_pre"]

    return out, wide

In [119]:
classifier_cols = [
    "xbow_topic_controversy",
    "topic_controversy",
    "llama_controversy",
    "factor_controversy"
]

all_wide = []

for col in classifier_cols:
    _, wide = tweet_topic_share_pre_post(notes_df_filter, col)
    wide["classifier"] = col
    all_wide.append(wide)

all_wide = pd.concat(all_wide, ignore_index=True)
display(all_wide)

,topic_type,percent_post,percent_pre,ci_lo_percent_post,ci_lo_percent_pre,ci_hi_percent_post,ci_hi_percent_pre,post_minus_pre_pp,classifier
0,controversial,34.668018,38.496583,32.599640,35.334705,36.795986,41.758687,-3.828565,xbow_topic_controversy
1,non_controversial,6.538267,6.150342,5.529729,4.744180,7.715723,7.938551,0.387925,xbow_topic_controversy
2,controversial,52.407501,53.075171,50.201229,49.767867,54.604416,56.355682,-0.667670,topic_controversy
3,non_controversial,25.443487,17.653759,23.570615,15.274203,27.411801,20.315137,7.789729,topic_controversy
4,controversial,50.025342,58.428246,47.821148,55.138418,52.229437,61.644642,-8.402904,llama_controversy
5,non_controversial,28.535226,15.945330,26.585797,13.672792,30.568080,18.514576,12.589895,llama_controversy
6,controversial,7.754688,2.164009,6.654900,1.389677,9.018667,3.355120,5.590679,factor_controversy
7,non_controversial,25.038013,6.833713,23.176094,5.345777,26.996950,8.697743,18.204300,factor_controversy
